In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

We can test different values later.

In [2]:
NUM_EPOCHS = 30

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

mps


Using CNN with depthwise separable convolution for paramater reduction.

In [4]:
class ConvNet(nn.Module):
  def __init__(self):
        super().__init__()
        # Input 94 x 80 x 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=0),  # 92 x 78 x 8
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)       # 46 x 39 x 8
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(8, 8, kernel_size=3, groups=8, padding=0), # 44 x 37 x 8
            nn.Conv2d(8, 12, kernel_size=1, bias=False), # Depthwise separable convolution
            nn.BatchNorm2d(12),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)               # 22 x 18 x 12
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(12, 12, kernel_size=3, groups=12, padding=0), # 20 x 16 x 12
            nn.Conv2d(12, 16, kernel_size=1, bias=False), # Depthwise separable convolution
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)                   # 10 x 8 x 16
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, groups=16, padding=0), # 8 x 6 x 16
            nn.Conv2d(16, 24, kernel_size=1),
            nn.BatchNorm2d(24),
            nn.ReLU()                                               # 8 x 6 x 24
        )

        # gap might be a bad idea
        # but maybe a smaller adaptive pooling layer will work better
        self.pool = nn.AdaptiveAvgPool2d((2, 2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(24 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 10),
        )

  def forward(self, x):
        # Input in (Batch, Channels, Height, Width)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

In [5]:
net = ConvNet().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

In [6]:
total_params = sum(p.numel() for p in net.parameters())
print(f"Total Parameters: {total_params:,}")

Total Parameters: 8,114


We make sure to extract just the green channel.

In [7]:
from transforms_file import extract_green_channel

data_transforms = transforms.Compose([
    transforms.ToTensor(), # We need normalized into 0-1.0
    transforms.Lambda(extract_green_channel),  # Keep only the green channel 
    transforms.Resize((94, 80)),               
])


train_dataset = datasets.ImageFolder(root="gcp_dataset/train/numbers", transform=data_transforms)
trainloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_dataset = datasets.ImageFolder(root="gcp_dataset/test/numbers", transform=data_transforms)
testloader = DataLoader(
    test_dataset,
    batch_size=32,
    num_workers=4,
    pin_memory=True
)

In [8]:
print("Train:", len(train_dataset), "images,", len(trainloader), "batches")
print("Test:", len(test_dataset), "images,", len(testloader), "batches")

Train: 2739 images, 86 batches
Test: 214 images, 7 batches


In [20]:
print("Train classes:", train_dataset.class_to_idx)
print("Test classes:", test_dataset.class_to_idx)
print("Train counts:", torch.bincount(torch.tensor(train_dataset.targets)))
print("Test counts:", torch.bincount(torch.tensor(test_dataset.targets)))

Train classes: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
Test classes: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
Train counts: tensor([174, 186, 165, 122, 360, 432, 458, 296, 288, 258])
Test counts: tensor([11, 10,  4, 13, 32, 37, 39, 23, 29, 16])


In [13]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()

        if batch % 15 == 0:
            loss, current = loss.item(), batch * 32 + len(X)
            print(f"{loss:>7f} [{current:>5d}/{size:>5d}]")

    # Using the scheduler to adjust the learning rate
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"LR: {current_lr:.6f}")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    torch.mps.synchronize()
    total_loss = torch.zeros((), device=device)
    total_correct = torch.zeros((), device=device)
    total = 0

    with torch.inference_mode():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            total_loss += loss_fn(logits, y) * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum()
            total += y.size(0)

    torch.mps.synchronize()
    print(
        f"Accuracy: {100 * (total_correct / total).item():.1f}%, "
        f"Loss: {(total_loss / total).item():.4f}\n"
    )

In [14]:
for n in range(NUM_EPOCHS):
    train_loop(trainloader, net, loss_fn, optimizer)
    test_loop(testloader, net, loss_fn)

print("Done!")

2.334065 [   32/ 2739]
2.130352 [  512/ 2739]
2.022077 [  992/ 2739]
1.766841 [ 1472/ 2739]
1.566842 [ 1952/ 2739]
1.383092 [ 2432/ 2739]
LR: 0.000997
Accuracy: 69.2%, Loss: 1.1362

1.086071 [   32/ 2739]
0.848624 [  512/ 2739]
0.757486 [  992/ 2739]
0.508894 [ 1472/ 2739]
0.493055 [ 1952/ 2739]
0.418345 [ 2432/ 2739]
LR: 0.000989
Accuracy: 92.5%, Loss: 0.3632

0.316676 [   32/ 2739]
0.308807 [  512/ 2739]
0.231278 [  992/ 2739]
0.233258 [ 1472/ 2739]
0.235004 [ 1952/ 2739]
0.065998 [ 2432/ 2739]
LR: 0.000976
Accuracy: 99.1%, Loss: 0.0999

0.087007 [   32/ 2739]
0.117954 [  512/ 2739]
0.069284 [  992/ 2739]
0.196437 [ 1472/ 2739]
0.052773 [ 1952/ 2739]
0.073347 [ 2432/ 2739]
LR: 0.000957
Accuracy: 99.5%, Loss: 0.0418

0.093183 [   32/ 2739]
0.084125 [  512/ 2739]
0.034957 [  992/ 2739]
0.111905 [ 1472/ 2739]
0.033419 [ 1952/ 2739]
0.021830 [ 2432/ 2739]
LR: 0.000933
Accuracy: 99.5%, Loss: 0.0413

0.046700 [   32/ 2739]
0.055126 [  512/ 2739]
0.054556 [  992/ 2739]
0.030281 [ 1472/ 2739

In [15]:
val_dataset = datasets.ImageFolder(root="gcp_dataset/valid/numbers", transform=data_transforms)
valloader = DataLoader(
    val_dataset,
    batch_size=32,
    num_workers=4,
    pin_memory=True
)

In [16]:
test_loop(valloader, net, loss_fn)

Accuracy: 99.4%, Loss: 0.0420



In [17]:
images, labels = next(iter(trainloader))

# 3. Extract just the first image from that batch
# Shape transitions from [Batch, C, H, W] to [1, C, H, W]
single_image = images[0].unsqueeze(0)
single_label = labels[0]  # If you want to check the ground truth

# 4. Move to device and run inference
single_image = single_image.to(device)

with torch.no_grad():  # Disables gradient calculation to save memory
    output = net(single_image)
    print(output)
    print(single_label)  # Print the ground truth label for comparison

tensor([[ -1.6937,  -5.3010, -17.0064,  -4.7023,  -2.5227,  -0.9340,  11.3271,
          -6.3584,   0.6992,  -9.5441]], device='mps:0')
tensor(6)


In [18]:
exported_program = torch.export.export(net, args=(torch.randn(1, 1, 94, 80).to(device),))
torch.export.save(exported_program, "team_air_gcp_localization_protoype.pt2")

In [19]:
torch.save(net.state_dict(), "team_air_gcp_localization_protoype.pth")